see [the README](../../README.md) for information about how to run this notebook with the right dependencies installed.

# Preprocessing glass plates with YOLOv11

This notebook explains how YOLOv11 is used as a preprocessing method for the digitized glass plates ('glasplaten' in Dutch). 
The main problem that the preprocessing solves is that many of the glass plate scans contain a border with a description of the image (as can be seen in the image below). \
While the description contains valuable information for extracting metadata, during clustering, we saw that it divided the images in two large sets: "with border" and "without border".
Though we don't want the clustering to take this distinction into account.

Therefore, we needed this preprocessing step, that gets the actual photographic content from the glass plate:

<img src="./example.webp" width="500">

We quickly established that conventional preprocessing-methodes would require a lot of tweaking, since the size of the border and description contents would change frequently. Therefore, we opted to train an object-detection model that could detect the actual image inside the full scan. This is merely a preprocessing step during clustering that won't propagate throughout the rest of the pipeline. \
However, the output of this processing could also guide us to which scans might need HTR/OCR (e.g. when the border is large, try to read text in it or something).

We used the YoloV11m(medium) variant to train 350 of our images (both images with and without border) for 40 epochs. The 350 images were annotated in LabelStudio (LS). 

The following code shows how to use the YOLO model for preprocessing. \
To run the pretrained model of the Glashelder project, you need to have access to the `best.pt` file. \
It is located in the glashelder share, at `models/best.pt`.

If you do not have access to that share (only affiliates of the project will have access),
you can find other pre-trained yolo model weights online, for example [this ultralytics github release page](https://github.com/ultralytics/assets/releases/tag/v8.3.0). \
These models aren't trained to perform the exact task we are trying to solve here, however. \
In case you download one of the ultralytics pretrained yolo models, this notebook will crop to the first 'class 0' crop of the image, 
which on those models corresponds to the first person, if any.

The model we trained takes a glass plate image, and outputs a crop of the photographic content in it.

## Inference on a set of images with the pretrained Glashelder model

Load the necessary python libraries:

In [ ]:
from preprocessing.yolo_cropper import run_crop_on_batch_of_images as rcbi

Adapt the variables in the code block below with your own filepaths.

- MODEL: path to a `.pt` file. This can be a filepath to a locally stored .pt file or one you pull from a website such as huggingface. 
- INPUT_FOLDER: filepath to your folder with images that need to be processed.
- OUTPUT_FOLDER = filepath to the folder you use for the output of the model (the cropped images).

In [ ]:
MODEL = "/path/to/the/pt_file/best.pt"  # MODIFY ME
INPUT_FOLDER = "/path/to/input/folder"  # MODIFY ME
OUTPUT_FOLDER = "/path/to/output/folder"  # MODIFY ME

Run the following code block to run the model on your images:

In [ ]:
rcbi(MODEL, INPUT_FOLDER, OUTPUT_FOLDER)
print(f"Images are saved at '{OUTPUT_FOLDER}'!")

### Compare input and output images

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

input_images = sorted(Path(INPUT_FOLDER).glob("*.jpg"))[:3]

fig, axes = plt.subplots(len(input_images), 2, figsize=(10, 5 * len(input_images)))
if len(input_images) == 1:
    axes = [axes]

for ax_row, img_path in zip(axes, input_images, strict=True):
    output_path = Path(OUTPUT_FOLDER) / img_path.name

    ax_row[0].imshow(Image.open(img_path))
    ax_row[0].set_title(f"Input: {img_path.name}")
    ax_row[0].axis("off")

    if output_path.exists():
        ax_row[1].imshow(Image.open(output_path))
        ax_row[1].set_title(f"Output: {output_path.name}")
    else:
        ax_row[1].text(0.5, 0.5, "No output found", ha="center", va="center")
        ax_row[1].set_title("Output: not found")
    ax_row[1].axis("off")

plt.tight_layout()
plt.show()

## Creating a similar model (or with additional data).

### Annotating images with LabelStudio

We created our dataset using [LabelStudio](https://labelstud.io).
You can find documentation on their website about how to export annotations in YOLO format, which is what we did.

### Getting training data

Within the folder where this notebook lives, several folders with images and .txt files need to be created for the training data.

The directory you need to create should have this structure:

```raw
ground_truth_dir
├── classes.txt
├── images
│   ├── image_00.png
│   ├── image_01.png
│   └── ...
└── labels
    ├── image_00.txt
    ├── image_01.txt
    └── ...
```

**Create the following:**
- Folder called `ground_truth_dir`


- .txt file called `classes`
    
    The `classes.txt` file should contain one line of text, eg. "content". In general, a YOLO model can be trained to detect and crop multiple classes of objects in images. However, for this use case of only cropping images to their content, we will be training the model to just detect and crop one class.


- Folder called `images`
    
    In the `images/` directory, a set of images need to be present.


- Folder called `labels`
    
    In the `labels/` directory, .txt files with the same file names as in `images` should be present. These are the boundary annotations on the images, which are the rectangular regions which we want to keep in the images.
    
    To gain the best results you should annotate the images in a tool such as LabelStudio or use the values from the example below.

    The format of these `.txt` files is the following:

    ```txt
    class_id c_x c_y w h
    ```

    - `class_id` will always be `0`, since we have just one class.
    - `c_x` and `c_y` are the location in the image of the center of the cropped region, as fractions of the full image width / height.
    - `w` and `h` are the width and height of the image, also as fractions of the full image height.

    for example: 

    ```image_00.txt
    0 0.45 0.52 0.9 0.91
    ```
    



### Creating the dataset. 

This section assumes that you have the exported data in the format explained above.

Load the necessary python libraries:

In [29]:
from preprocessing.dataset_prep import create_yolo_dataset

Adapt the variables in the code block below where necessary. 

You can provide the split between the training and validation set (which is 80%-20% by default). 

You can provide the seed which is used to randomize the training and validation set. A seed is given to make a model behave in the same manner each time. This is important for reproducability.

- GROUND_TRUTH_FOLDER = `./ground_truth"` => leads to a ground_truth folder in the folder where you put this notebook
- OUTPUT_FOLDER = `./output` => this leads to an output folder in the folder where you put this notebook
- TRAIN_TEST_SPLIT = `0.8` => here the training data is set to 80%
- SEED = `50` => here the seed is set to 50.



In [ ]:
GROUND_TRUTH_FOLDER = "./ground_truth_dir"  # MODIFY ME IF NECESSARY
OUTPUT_FOLDER = "./output"  # MODIFY ME IF NECESSARY
TRAIN_TEST_SPLIT = 0.8  # MODIFY ME IF NECESSARY
SEED = 50  # MODIFY ME IF NECESSARY

In [33]:
create_yolo_dataset(GROUND_TRUTH_FOLDER, OUTPUT_FOLDER, TRAIN_TEST_SPLIT, SEED)
print(f"DATASET is saved at '{OUTPUT_FOLDER}'!")

## Training the model

This sections provides the function for training the YOLOv11 model (which is very close to the actual function provided by UltraLytics). 

The image size is set to 1028x1028. 

In [ ]:
# Possible models: yolo11n.pt yolo11s.pt yolo11m.pt yolo11l.pt yolo11x.pt. You can also use our fine-tuned model as a starting point.
# NOTE: you don't need to download this .pt file yourself. The Yolo library will download it for you when it isn't already present
# on your system
MODEL = "yolo11m.pt"
DATA_YAML = "./data.yaml"  # data.yaml can be found in the DATASET, at OUTPUT_FOLDER of the previous step.
EPOCHS = 20
OUTPUT_FOLDER = "./output"
NEW_MODEL_NAME = "preprocess_model"  # WARNING: if a model already has the same name in the folder, it will be overwritten

In [35]:
from preprocessing.yolo_train import train_yolo_model

In [36]:
train_yolo_model(MODEL, DATA_YAML, EPOCHS, OUTPUT_FOLDER, NEW_MODEL_NAME)